# GAN Fundamentals: Adversarial Training from Scratch

**What we'll build:** A complete understanding of Generative Adversarial Networks, from theory to implementation.

**Why it matters:** GANs revolutionized generative modeling by framing generation as a game between two networks. Understanding GANs deeply reveals fundamental insights about implicit density estimation, adversarial training dynamics, and why neural networks can be so powerful at generation.

**What intuitions we'll develop:**
- How the Generator and Discriminator compete and improve each other
- The minimax game formulation and what "equilibrium" means
- Why GANs produce sharper images than VAEs
- Common failure modes (mode collapse, training instability) and how to fix them
- Different GAN loss functions and when to use each

## Learning Approach

We'll build understanding incrementally:
1. **Theory**: The adversarial game formulation
2. **Architecture**: Generator and Discriminator design
3. **Training Loop**: The delicate dance of alternating updates
4. **Loss Functions**: Original, Non-saturating, Wasserstein, and more
5. **Failure Modes**: Mode collapse, vanishing gradients, instability
6. **Training Tricks**: Practical techniques that make GANs work
7. **Hands-On**: Train a DCGAN on MNIST

---
## 1. Setup

Import necessary libraries and configure the environment.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from aiml_notebooks import get_device, set_seed

%load_ext autoreload
%autoreload 2

### Configuration

All hyperparameters in one place for easy experimentation.

In [ ]:
CONFIG = {
    # Reproducibility
    'seed': 42,  # Random seed for reproducibility
    
    # Data
    'batch_size': 128,  # Number of samples per training batch
    'image_size': 28,  # Image dimensions (MNIST is 28x28)
    'num_channels': 1,  # Number of image channels (1 for grayscale)
    
    # Model
    'latent_dim': 100,  # Size of latent noise vector z
    'g_features': 64,  # Base feature maps in generator
    'd_features': 64,  # Base feature maps in discriminator
    
    # Training
    'learning_rate': 0.0002,  # Adam learning rate (DCGAN paper)
    'beta1': 0.5,  # Adam beta1 (DCGAN paper recommends 0.5)
    'beta2': 0.999,  # Adam beta2
    'num_epochs': 25,  # Number of training epochs
    'n_critic': 1,  # Discriminator updates per generator update
    
    # Tricks
    'label_smoothing': True,  # Use soft labels (0.9 instead of 1.0)
    'add_noise': False,  # Add noise to discriminator inputs
}

Set random seed for reproducibility and configure device.

In [ ]:
set_seed(CONFIG['seed'])
device = get_device()
print(f"Using device: {device}")

---
## 2. The GAN Framework: A Game-Theoretic View

### The Core Insight

Instead of explicitly modeling the data distribution $P_{data}(x)$, GANs set up a **game** between two neural networks:

| Player | Role | Goal |
|--------|------|------|
| **Generator (G)** | Forger | Create fake data that looks real |
| **Discriminator (D)** | Detective | Distinguish real from fake data |

### The Training Dynamic

1. **G samples noise** $z \sim P_z(z)$ (usually Gaussian)
2. **G generates fake data** $\tilde{x} = G(z)$
3. **D sees both real and fake** and tries to classify them
4. **Both learn from D's feedback**:
   - D improves at detecting fakes
   - G improves at fooling D

### Why This Works

The key insight: **D provides a learning signal to G**.

- Without D, G has no way to know if its outputs are realistic
- D acts as a learned loss function that adapts as G improves
- Competition drives both networks to improve

### The Minimax Game

The original GAN objective is a **minimax game**:

$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

**Breaking it down:**

| Term | Meaning | D wants | G wants |
|------|---------|---------|----------|
| $\log D(x)$ | Log-prob that real data is real | Maximize (D(x)→1) | N/A |
| $\log(1-D(G(z)))$ | Log-prob that fake data is fake | Maximize (D(G(z))→0) | Minimize (D(G(z))→1) |

**D maximizes** the full expression (correctly classify everything).

**G minimizes** only the second term (fool D into thinking fakes are real).

### Nash Equilibrium

At the **optimal solution** (Nash equilibrium):

1. **$P_G = P_{data}$**: Generator's distribution matches the real data
2. **$D(x) = 0.5$ for all x**: Discriminator can't tell real from fake

**Intuition**: If G perfectly matches the data distribution, there's literally no information D can use to distinguish them.

### Why GANs Produce Sharp Images

Unlike VAEs (which use MSE loss and tend to average):
- GANs don't directly penalize pixel differences
- D learns to detect "unrealistic" features
- G must produce convincing details to fool D
- Result: Sharp, realistic outputs

### Visualizing the Adversarial Game

Let's visualize how G and D interact during training.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Stage 1: Early training
ax = axes[0]
x = np.linspace(-3, 3, 100)
real_dist = np.exp(-0.5 * (x - 1)**2) / np.sqrt(2 * np.pi)
fake_dist = np.exp(-0.5 * (x + 1.5)**2 * 0.5) / np.sqrt(2 * np.pi) * 0.8
ax.fill_between(x, real_dist, alpha=0.5, color='blue', label='Real data')
ax.fill_between(x, fake_dist, alpha=0.5, color='red', label='Generated')
ax.axhline(y=0.5, color='green', linestyle='--', alpha=0.5)
ax.set_title('Early Training\nG produces clearly different samples', fontweight='bold')
ax.legend()
ax.set_ylim(0, 0.6)

# Stage 2: Mid training
ax = axes[1]
fake_dist = np.exp(-0.5 * (x - 0)**2 * 0.7) / np.sqrt(2 * np.pi) * 0.9
ax.fill_between(x, real_dist, alpha=0.5, color='blue', label='Real data')
ax.fill_between(x, fake_dist, alpha=0.5, color='red', label='Generated')
ax.set_title('Mid Training\nG getting closer to real distribution', fontweight='bold')
ax.legend()
ax.set_ylim(0, 0.6)

# Stage 3: Convergence
ax = axes[2]
fake_dist = np.exp(-0.5 * (x - 1)**2) / np.sqrt(2 * np.pi) * 0.95
ax.fill_between(x, real_dist, alpha=0.5, color='blue', label='Real data')
ax.fill_between(x, fake_dist, alpha=0.5, color='red', label='Generated')
ax.set_title('Convergence\n$P_G \\approx P_{data}$, D outputs ~0.5', fontweight='bold')
ax.legend()
ax.set_ylim(0, 0.6)

plt.tight_layout()
plt.show()

print("Key insight: G learns to match the data distribution through adversarial feedback from D")

---
## 3. GAN Loss Functions

Different loss formulations have different training dynamics. Let's explore the main variants.

### 3.1 Original GAN Loss (Minimax)

**Discriminator Loss:**
$$L_D = -\mathbb{E}_{x}[\log D(x)] - \mathbb{E}_{z}[\log(1 - D(G(z)))]$$

**Generator Loss:**
$$L_G = \mathbb{E}_{z}[\log(1 - D(G(z)))]$$

**Problem**: When D is very confident (D(G(z)) → 0), the gradient for G vanishes!

$$\frac{\partial}{\partial \theta_G} \log(1 - D(G(z))) \approx 0 \text{ when } D(G(z)) \approx 0$$

### 3.2 Non-Saturating GAN Loss

**Solution**: Instead of minimizing $\log(1 - D(G(z)))$, maximize $\log D(G(z))$:

**Generator Loss (modified):**
$$L_G = -\mathbb{E}_{z}[\log D(G(z))]$$

**Same equilibrium, better gradients!**

When D(G(z)) is small, $-\log D(G(z))$ is large, providing strong gradient signal.

This is the **default choice** in most GAN implementations.

### Comparing Gradient Signals

Let's visualize why non-saturating loss works better.

In [ ]:
d_fake = np.linspace(0.001, 0.999, 100)  # D(G(z)) output

# Original (minimax) loss for G
original_loss = np.log(1 - d_fake)
original_grad = -1 / (1 - d_fake)  # Derivative

# Non-saturating loss for G
ns_loss = -np.log(d_fake)
ns_grad = -1 / d_fake  # Derivative

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss values
ax = axes[0]
ax.plot(d_fake, original_loss, 'b-', linewidth=2, label='Original: log(1-D(G(z)))')
ax.plot(d_fake, ns_loss, 'r-', linewidth=2, label='Non-saturating: -log(D(G(z)))')
ax.set_xlabel('D(G(z))', fontsize=12)
ax.set_ylabel('Generator Loss', fontsize=12)
ax.set_title('Loss Value vs Discriminator Output', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)

# Gradient magnitude
ax = axes[1]
ax.plot(d_fake, np.abs(original_grad), 'b-', linewidth=2, label='Original gradient')
ax.plot(d_fake, np.abs(ns_grad), 'r-', linewidth=2, label='Non-saturating gradient')
ax.set_xlabel('D(G(z))', fontsize=12)
ax.set_ylabel('|Gradient|', fontsize=12)
ax.set_title('Gradient Magnitude vs Discriminator Output', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 20)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("Key insight:")
print("• Original loss: Gradient vanishes when D(G(z))→0 (early training!)")
print("• Non-saturating: Strong gradient when D(G(z))→0 (exactly when G needs it!)")

### 3.3 Wasserstein GAN (WGAN) Loss

WGAN uses a completely different distance metric: **Earth Mover's Distance (Wasserstein-1)**.

**Critic Loss** (D is called "Critic" in WGAN):
$$L_C = \mathbb{E}_{z}[D(G(z))] - \mathbb{E}_{x}[D(x)]$$

**Generator Loss:**
$$L_G = -\mathbb{E}_{z}[D(G(z))]$$

**Key differences:**
- No sigmoid on D's output (outputs unbounded score, not probability)
- Requires **Lipschitz constraint** on D (weight clipping or gradient penalty)
- More stable training, meaningful loss curves

**Gradient Penalty (WGAN-GP):**
$$L_C = \mathbb{E}_{z}[D(G(z))] - \mathbb{E}_{x}[D(x)] + \lambda \mathbb{E}_{\hat{x}}[(\|\nabla_\hat{x} D(\hat{x})\|_2 - 1)^2]$$

Where $\hat{x}$ is interpolated between real and fake samples.

### 3.4 Least Squares GAN (LSGAN) Loss

Uses MSE instead of cross-entropy:

**Discriminator Loss:**
$$L_D = \frac{1}{2}\mathbb{E}_{x}[(D(x) - 1)^2] + \frac{1}{2}\mathbb{E}_{z}[D(G(z))^2]$$

**Generator Loss:**
$$L_G = \frac{1}{2}\mathbb{E}_{z}[(D(G(z)) - 1)^2]$$

**Benefits:**
- More stable than original GAN
- Penalizes samples far from decision boundary
- Smoother gradients

### Loss Function Comparison

| Loss | D Output | Stability | When to Use |
|------|----------|-----------|-------------|
| **Original (Minimax)** | Sigmoid | Poor | Historical reference only |
| **Non-Saturating** | Sigmoid | Good | Default choice |
| **WGAN** | Linear | Very good | When stability is critical |
| **WGAN-GP** | Linear | Excellent | Production systems |
| **LSGAN** | Linear | Good | Simple alternative to WGAN |
| **Hinge** | Linear | Very good | Large-scale (BigGAN, StyleGAN) |

**Our choice**: Non-saturating BCE loss (most common, good balance of simplicity and stability).

### Implementing Different Loss Functions

Let's implement several loss variants for comparison.

In [ ]:
def bce_loss_d(d_real, d_fake, label_smoothing=False):
    """
    Binary Cross-Entropy loss for Discriminator.
    Non-saturating formulation.
    """
    batch_size = d_real.size(0)
    
    # Labels
    real_label = 0.9 if label_smoothing else 1.0  # Label smoothing trick
    real_labels = torch.full((batch_size, 1), real_label, device=d_real.device)
    fake_labels = torch.zeros(batch_size, 1, device=d_fake.device)
    
    # Losses
    loss_real = F.binary_cross_entropy(d_real, real_labels)
    loss_fake = F.binary_cross_entropy(d_fake, fake_labels)
    
    return loss_real + loss_fake


def bce_loss_g(d_fake):
    """
    Binary Cross-Entropy loss for Generator.
    Non-saturating: maximize log(D(G(z))) instead of minimize log(1-D(G(z))).
    """
    batch_size = d_fake.size(0)
    real_labels = torch.ones(batch_size, 1, device=d_fake.device)
    return F.binary_cross_entropy(d_fake, real_labels)


def wasserstein_loss_d(d_real, d_fake):
    """
    Wasserstein loss for Discriminator (Critic).
    """
    return d_fake.mean() - d_real.mean()


def wasserstein_loss_g(d_fake):
    """
    Wasserstein loss for Generator.
    """
    return -d_fake.mean()


def lsgan_loss_d(d_real, d_fake):
    """
    Least Squares loss for Discriminator.
    """
    loss_real = 0.5 * ((d_real - 1) ** 2).mean()
    loss_fake = 0.5 * (d_fake ** 2).mean()
    return loss_real + loss_fake


def lsgan_loss_g(d_fake):
    """
    Least Squares loss for Generator.
    """
    return 0.5 * ((d_fake - 1) ** 2).mean()


print("Loss functions implemented:")
print("  • BCE (Non-saturating) - default choice")
print("  • Wasserstein - for stability")
print("  • Least Squares - simple alternative")

---
## 4. GAN Architecture: Generator & Discriminator

### DCGAN Guidelines

The DCGAN paper (Radford et al., 2015) established architecture guidelines that significantly improve GAN training:

1. **Replace pooling with strided convolutions**
   - D: strided conv for downsampling
   - G: transposed conv for upsampling

2. **Use batch normalization** in both G and D
   - Exception: Don't use on G's output layer
   - Exception: Don't use on D's input layer

3. **Remove fully connected hidden layers** for deeper architectures

4. **Activations:**
   - G: ReLU (hidden), Tanh (output)
   - D: LeakyReLU (all layers)

### Generator Architecture

The Generator transforms random noise into images:

```
z (100D noise) → Project → Reshape → ConvTranspose → ... → Image
```

In [ ]:
class Generator(nn.Module):
    """
    DCGAN Generator: z → image
    
    Architecture:
    - Project noise to feature maps
    - Upsample through transposed convolutions
    - Output with Tanh (matches [-1, 1] normalization)
    """
    
    def __init__(self, latent_dim=100, num_channels=1, features=64):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Project and reshape: z → (features*4, 7, 7)
        self.project = nn.Sequential(
            nn.Linear(latent_dim, features * 4 * 7 * 7),
            nn.BatchNorm1d(features * 4 * 7 * 7),
            nn.ReLU(True)
        )
        
        # Upsampling convolutions
        self.conv_blocks = nn.Sequential(
            # (features*4, 7, 7) → (features*2, 14, 14)
            nn.ConvTranspose2d(features * 4, features * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features * 2),
            nn.ReLU(True),
            
            # (features*2, 14, 14) → (features, 28, 28)
            nn.ConvTranspose2d(features * 2, features, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features),
            nn.ReLU(True),
            
            # (features, 28, 28) → (num_channels, 28, 28)
            nn.Conv2d(features, num_channels, 3, 1, 1, bias=False),
            nn.Tanh()  # Output in [-1, 1]
        )
        
        self.features = features
    
    def forward(self, z):
        # Project and reshape
        x = self.project(z)
        x = x.view(x.size(0), self.features * 4, 7, 7)
        
        # Generate image
        return self.conv_blocks(x)


# Test generator
G = Generator(latent_dim=CONFIG['latent_dim'], features=CONFIG['g_features']).to(device)
z_test = torch.randn(4, CONFIG['latent_dim'], device=device)
fake_test = G(z_test)

print(f"Generator:")
print(f"  Input: z shape = {z_test.shape}")
print(f"  Output: image shape = {fake_test.shape}")
print(f"  Parameters: {sum(p.numel() for p in G.parameters()):,}")

### Discriminator Architecture

The Discriminator classifies images as real or fake:

```
Image → Conv (stride) → ... → Flatten → Probability
```

In [ ]:
class Discriminator(nn.Module):
    """
    DCGAN Discriminator: image → real/fake probability
    
    Architecture:
    - Strided convolutions for downsampling
    - LeakyReLU activations throughout
    - Sigmoid output for probability
    """
    
    def __init__(self, num_channels=1, features=64):
        super().__init__()
        
        self.conv_blocks = nn.Sequential(
            # (num_channels, 28, 28) → (features, 14, 14)
            nn.Conv2d(num_channels, features, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            # (features, 14, 14) → (features*2, 7, 7)
            nn.Conv2d(features, features * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features * 2),
            nn.LeakyReLU(0.2, inplace=True),
            
            # (features*2, 7, 7) → (features*4, 3, 3)
            nn.Conv2d(features * 2, features * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features * 4),
            nn.LeakyReLU(0.2, inplace=True),
        )
        
        # Output layer
        self.output = nn.Sequential(
            nn.Flatten(),
            nn.Linear(features * 4 * 3 * 3, 1),
            nn.Sigmoid()  # Probability output
        )
    
    def forward(self, x):
        features = self.conv_blocks(x)
        return self.output(features)


# Test discriminator
D = Discriminator(features=CONFIG['d_features']).to(device)
d_out = D(fake_test)

print(f"Discriminator:")
print(f"  Input: image shape = {fake_test.shape}")
print(f"  Output: probability shape = {d_out.shape}")
print(f"  Parameters: {sum(p.numel() for p in D.parameters()):,}")

### Weight Initialization

DCGAN recommends initializing weights from $\mathcal{N}(0, 0.02)$.

In [ ]:
def weights_init(m):
    """
    Initialize weights according to DCGAN paper.
    Conv and BatchNorm layers: N(0, 0.02)
    """
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# Apply to both networks
G.apply(weights_init)
D.apply(weights_init)
print("Weights initialized with N(0, 0.02)")

---
## 5. The Training Loop: Alternating Optimization

GAN training alternates between updating D and G. The order and frequency matter!

### Standard Training Algorithm

```
for each batch:
    # Step 1: Update Discriminator
    1a. Sample real batch x from data
    1b. Sample noise z, generate fake batch G(z)
    1c. Compute D loss on real and fake
    1d. Update D parameters
    
    # Step 2: Update Generator
    2a. Sample new noise z
    2b. Generate fake batch G(z)
    2c. Compute G loss (fool D)
    2d. Update G parameters
```

**Critical**: When updating G, we don't update D (and vice versa)!

### Implementing the Training Steps

In [ ]:
def train_discriminator(D, G, optimizer_D, real_images, latent_dim, device, label_smoothing=True):
    """
    Train Discriminator for one step.
    
    Returns: d_loss, D(real), D(fake)
    """
    batch_size = real_images.size(0)
    
    # Zero gradients
    optimizer_D.zero_grad()
    
    # --- Real images ---
    d_real = D(real_images)
    
    # --- Fake images ---
    z = torch.randn(batch_size, latent_dim, device=device)
    fake_images = G(z).detach()  # Detach: don't backprop through G
    d_fake = D(fake_images)
    
    # --- Compute loss and update ---
    d_loss = bce_loss_d(d_real, d_fake, label_smoothing=label_smoothing)
    d_loss.backward()
    optimizer_D.step()
    
    return d_loss.item(), d_real.mean().item(), d_fake.mean().item()


def train_generator(D, G, optimizer_G, batch_size, latent_dim, device):
    """
    Train Generator for one step.
    
    Returns: g_loss
    """
    # Zero gradients
    optimizer_G.zero_grad()
    
    # Generate fake images
    z = torch.randn(batch_size, latent_dim, device=device)
    fake_images = G(z)
    
    # Get discriminator's opinion (want it to think they're real)
    d_fake = D(fake_images)
    
    # Compute loss and update
    g_loss = bce_loss_g(d_fake)
    g_loss.backward()
    optimizer_G.step()
    
    return g_loss.item()


print("Training functions defined!")
print("\nKey points:")
print("  • D training: detach fake images (don't update G)")
print("  • G training: don't detach (gradients flow through D to G)")
print("  • Separate optimizers for G and D")

---
## 6. Failure Modes and Training Tricks

GANs are notoriously difficult to train. Let's understand the common failure modes and how to fix them.

### 6.1 Mode Collapse

**What it is**: Generator produces limited variety, often just one or few "modes" of the data.

**Example**: Training on digits, G only produces "1" because D learned to accept it.

**Why it happens**:
- G finds a "safe" output that fools D
- D adapts to that output
- G collapses to another safe output
- Cycle continues, diversity never emerges

**Solutions**:
1. **Minibatch discrimination**: D considers relationships within batch
2. **Unrolled GANs**: G considers future D updates
3. **Feature matching**: Match statistics instead of fooling D
4. **Different architectures**: Progressive growing, StyleGAN

### Visualizing Mode Collapse

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# True data distribution (multiple modes)
x = np.linspace(-4, 4, 200)
true_dist = 0.3 * np.exp(-0.5 * (x + 2)**2) + 0.4 * np.exp(-0.5 * x**2) + 0.3 * np.exp(-0.5 * (x - 2)**2)

# Healthy generation
ax = axes[0]
healthy = 0.25 * np.exp(-0.5 * (x + 2)**2 * 1.2) + 0.35 * np.exp(-0.5 * x**2 * 0.9) + 0.35 * np.exp(-0.5 * (x - 2)**2 * 1.1)
ax.fill_between(x, true_dist, alpha=0.3, color='blue', label='Real data')
ax.fill_between(x, healthy, alpha=0.5, color='green', label='Generated')
ax.set_title('Healthy: Covers all modes', fontsize=12, fontweight='bold')
ax.legend()
ax.set_ylim(0, 0.6)

# Partial collapse
ax = axes[1]
partial = 0.6 * np.exp(-0.5 * x**2 * 0.8) + 0.15 * np.exp(-0.5 * (x - 2)**2 * 1.5)
ax.fill_between(x, true_dist, alpha=0.3, color='blue', label='Real data')
ax.fill_between(x, partial, alpha=0.5, color='orange', label='Generated')
ax.set_title('Partial Collapse: Missing mode', fontsize=12, fontweight='bold')
ax.legend()
ax.set_ylim(0, 0.6)

# Full collapse
ax = axes[2]
collapsed = 0.9 * np.exp(-0.5 * x**2 * 2)
ax.fill_between(x, true_dist, alpha=0.3, color='blue', label='Real data')
ax.fill_between(x, collapsed, alpha=0.5, color='red', label='Generated')
ax.set_title('Full Collapse: Single mode only', fontsize=12, fontweight='bold')
ax.legend()
ax.set_ylim(0, 0.6)

plt.tight_layout()
plt.show()

print("Detection: If generated samples lack diversity, mode collapse is likely.")

### 6.2 Vanishing Gradients (D too strong)

**What it is**: D becomes so good that G gets no useful gradient signal.

**Symptoms**:
- D(real) ≈ 1.0, D(fake) ≈ 0.0 consistently
- G loss stays high but doesn't decrease
- Generated images don't improve

**Solutions**:
1. **Non-saturating loss** (use -log(D(G(z))) instead of log(1-D(G(z))))
2. **Label smoothing**: Use 0.9 instead of 1.0 for real labels
3. **Instance noise**: Add noise to D's inputs
4. **Update ratio**: Train D less frequently
5. **Weaker D architecture**: Reduce D's capacity

### 6.3 Training Instability

**What it is**: Loss oscillates wildly, training diverges.

**Symptoms**:
- Losses don't converge, jump around
- Generated quality fluctuates
- May produce NaN losses

**Solutions**:
1. **Lower learning rate**: Try 0.0001 or 0.0002
2. **Adam β₁ = 0.5**: DCGAN recommendation
3. **Gradient clipping**: Prevent exploding gradients
4. **Spectral normalization**: Constrains D's Lipschitz constant
5. **WGAN-GP**: More stable loss formulation

### Summary of Training Tricks

| Trick | Problem it Solves | How it Works |
|-------|-------------------|---------------|
| **Label smoothing** | D too confident | Real labels: 0.9 instead of 1.0 |
| **Instance noise** | D too good early | Add Gaussian noise to D inputs |
| **One-sided label smoothing** | Mode collapse | Only smooth real labels, not fake |
| **Spectral norm** | Training instability | Normalize D weights by spectral norm |
| **Two-timescale** | Convergence issues | Different LR for G and D |
| **Progressive growing** | High-res instability | Start small, grow resolution |
| **Feature matching** | Mode collapse | Match D's intermediate features |
| **Minibatch discrimination** | Mode collapse | D sees batch statistics |

---
## 7. Hands-On: Training a DCGAN on MNIST

Let's put everything together and train a GAN!

### Load MNIST Dataset

In [ ]:
# Transform: normalize to [-1, 1] (matches Tanh output)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load MNIST
train_dataset = datasets.MNIST(
    root='./tmp/data',
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=0,
    drop_last=True  # Avoid issues with last incomplete batch
)

print(f"Dataset: MNIST")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Batches per epoch: {len(train_loader)}")

### Visualize Real Data

In [ ]:
def show_images(images, title="Images", nrow=8):
    """Display a grid of images."""
    # Denormalize from [-1, 1] to [0, 1]
    images = images * 0.5 + 0.5
    grid = make_grid(images, nrow=nrow, padding=2, normalize=False)
    
    plt.figure(figsize=(12, 6))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Show some real samples
real_batch, _ = next(iter(train_loader))
show_images(real_batch[:64], "Real MNIST Samples")

### Initialize Models and Optimizers

In [ ]:
# Create fresh models
G = Generator(
    latent_dim=CONFIG['latent_dim'],
    num_channels=CONFIG['num_channels'],
    features=CONFIG['g_features']
).to(device)

D = Discriminator(
    num_channels=CONFIG['num_channels'],
    features=CONFIG['d_features']
).to(device)

# Apply weight initialization
G.apply(weights_init)
D.apply(weights_init)

# Optimizers (DCGAN paper settings)
optimizer_G = optim.Adam(G.parameters(), lr=CONFIG['learning_rate'], betas=(CONFIG['beta1'], CONFIG['beta2']))
optimizer_D = optim.Adam(D.parameters(), lr=CONFIG['learning_rate'], betas=(CONFIG['beta1'], CONFIG['beta2']))

# Fixed noise for consistent visualization
fixed_noise = torch.randn(64, CONFIG['latent_dim'], device=device)

print("Models initialized:")
print(f"  Generator: {sum(p.numel() for p in G.parameters()):,} parameters")
print(f"  Discriminator: {sum(p.numel() for p in D.parameters()):,} parameters")
print(f"\nOptimizers: Adam with lr={CONFIG['learning_rate']}, β₁={CONFIG['beta1']}")

### Training Loop

Now the main event: training the GAN with careful monitoring.

In [ ]:
# Training history
history = {
    'g_loss': [],
    'd_loss': [],
    'd_real': [],
    'd_fake': []
}

print(f"Training GAN for {CONFIG['num_epochs']} epochs...\n")

for epoch in range(CONFIG['num_epochs']):
    G.train()
    D.train()
    
    epoch_g_loss = 0
    epoch_d_loss = 0
    epoch_d_real = 0
    epoch_d_fake = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']}", leave=False)
    
    for i, (real_images, _) in enumerate(pbar):
        real_images = real_images.to(device)
        batch_size = real_images.size(0)
        
        # --- Train Discriminator ---
        d_loss, d_real, d_fake = train_discriminator(
            D, G, optimizer_D, real_images,
            CONFIG['latent_dim'], device,
            label_smoothing=CONFIG['label_smoothing']
        )
        
        # --- Train Generator ---
        g_loss = train_generator(
            D, G, optimizer_G, batch_size,
            CONFIG['latent_dim'], device
        )
        
        # Accumulate metrics
        epoch_g_loss += g_loss
        epoch_d_loss += d_loss
        epoch_d_real += d_real
        epoch_d_fake += d_fake
        
        # Update progress bar
        pbar.set_postfix({
            'D_loss': f'{d_loss:.3f}',
            'G_loss': f'{g_loss:.3f}',
            'D(real)': f'{d_real:.3f}',
            'D(fake)': f'{d_fake:.3f}'
        })
    
    # Average metrics
    n_batches = len(train_loader)
    history['g_loss'].append(epoch_g_loss / n_batches)
    history['d_loss'].append(epoch_d_loss / n_batches)
    history['d_real'].append(epoch_d_real / n_batches)
    history['d_fake'].append(epoch_d_fake / n_batches)
    
    # Print epoch summary
    print(f"Epoch {epoch+1}/{CONFIG['num_epochs']} | "
          f"D_loss: {history['d_loss'][-1]:.4f} | G_loss: {history['g_loss'][-1]:.4f} | "
          f"D(real): {history['d_real'][-1]:.3f} | D(fake): {history['d_fake'][-1]:.3f}")
    
    # Visualize progress every 5 epochs
    if (epoch + 1) % 5 == 0 or epoch == 0:
        G.eval()
        with torch.no_grad():
            fake_images = G(fixed_noise)
        show_images(fake_images, f"Generated Images (Epoch {epoch+1})")

print("\nTraining complete!")

### Training Curves

Let's analyze the training dynamics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['g_loss']) + 1)

# Loss curves
ax = axes[0]
ax.plot(epochs, history['d_loss'], 'b-', linewidth=2, label='Discriminator Loss', marker='o')
ax.plot(epochs, history['g_loss'], 'r-', linewidth=2, label='Generator Loss', marker='s')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('GAN Training Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# D(real) and D(fake)
ax = axes[1]
ax.plot(epochs, history['d_real'], 'g-', linewidth=2, label='D(real)', marker='o')
ax.plot(epochs, history['d_fake'], 'orange', linewidth=2, label='D(fake)', marker='s')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Equilibrium')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Discriminator Output', fontsize=12)
ax.set_title('Discriminator Confidence', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("\nHealthy training indicators:")
print(f"  • D(real) should be 0.6-0.9: {history['d_real'][-1]:.3f}")
print(f"  • D(fake) should be 0.1-0.4: {history['d_fake'][-1]:.3f}")
print(f"  • Losses should be relatively balanced")

### Final Generated Samples

Let's see what our trained generator can produce!

In [ ]:
G.eval()

# Generate samples with different random seeds
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

for idx, ax in enumerate(axes.flat):
    with torch.no_grad():
        z = torch.randn(64, CONFIG['latent_dim'], device=device)
        fake = G(z)
        fake = fake * 0.5 + 0.5  # Denormalize
        grid = make_grid(fake, nrow=8, padding=2)
    
    ax.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    ax.set_title(f'Sample Batch {idx+1}', fontsize=12)
    ax.axis('off')

plt.suptitle('Generated MNIST Digits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Exploring the Latent Space

One fascinating aspect of GANs: the latent space often has meaningful structure.

### Latent Space Interpolation

We can smoothly transition between generated images by interpolating in latent space.

In [ ]:
def interpolate_latent(G, z1, z2, steps=10):
    """Interpolate between two latent vectors."""
    G.eval()
    
    with torch.no_grad():
        alphas = torch.linspace(0, 1, steps, device=z1.device)
        images = []
        
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            img = G(z)
            images.append(img)
        
        return torch.cat(images, dim=0)


# Perform several interpolations
fig, axes = plt.subplots(4, 1, figsize=(15, 10))

for i, ax in enumerate(axes):
    z1 = torch.randn(1, CONFIG['latent_dim'], device=device)
    z2 = torch.randn(1, CONFIG['latent_dim'], device=device)
    
    interp = interpolate_latent(G, z1, z2, steps=12)
    interp = interp * 0.5 + 0.5
    grid = make_grid(interp, nrow=12, padding=2)
    
    ax.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    ax.set_title(f'Interpolation {i+1}', fontsize=11)
    ax.axis('off')

plt.suptitle('Latent Space Interpolations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observation: Smooth transitions suggest a well-structured latent space!")

### Random Walk in Latent Space

Moving along a random direction reveals how the generator varies outputs.

In [ ]:
G.eval()

# Start point and random direction
z_base = torch.randn(1, CONFIG['latent_dim'], device=device)
direction = torch.randn(1, CONFIG['latent_dim'], device=device)
direction = direction / direction.norm()  # Normalize

# Walk along direction
steps = torch.linspace(-3, 3, 12, device=device)
walk_images = []

with torch.no_grad():
    for step in steps:
        z = z_base + step * direction
        img = G(z)
        walk_images.append(img)

walk_images = torch.cat(walk_images, dim=0)
walk_images = walk_images * 0.5 + 0.5

grid = make_grid(walk_images, nrow=12, padding=2)
plt.figure(figsize=(15, 3))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
plt.title('Random Walk in Latent Space (step: -3 to +3)', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print("Each direction in latent space corresponds to some visual variation!")

---
## Key Takeaways

### 1. The GAN Framework
- **Adversarial game**: Generator vs Discriminator
- **Implicit density**: No explicit $P(x)$, just learns to generate samples
- **Equilibrium**: $D(x) = 0.5$ when $P_G = P_{data}$

### 2. Loss Functions Matter
- **Original (minimax)**: Vanishing gradient problem
- **Non-saturating**: Better gradients, default choice
- **Wasserstein**: Most stable, but more complex

### 3. Common Failure Modes
- **Mode collapse**: Limited diversity in outputs
- **Vanishing gradients**: D too strong, G can't learn
- **Training instability**: Oscillating losses, divergence

### 4. Training Tricks Are Essential
- Label smoothing (0.9 instead of 1.0)
- DCGAN architecture guidelines
- Adam with β₁ = 0.5
- Proper weight initialization

### 5. The Training Loop
- Alternate between D and G updates
- Detach fake images when training D
- Monitor D(real) and D(fake) for health checks

---
## Exercises

1. **Loss comparison**: Implement LSGAN loss and compare training dynamics

2. **Architecture ablation**: What happens if you remove batch normalization?

3. **Induce mode collapse**: Train with n_critic=5 (update D 5x per G update). What happens?

4. **Conditional GAN**: Add class labels to generate specific digits

5. **Fashion-MNIST**: Apply the same architecture to Fashion-MNIST. Does it work as well?

6. **Latent dimension**: Try latent_dim = 10, 50, 200. How does it affect quality?

---
## Further Reading

**Foundational Papers:**
- Goodfellow et al. (2014): "Generative Adversarial Networks" - The original GAN paper
- Radford et al. (2015): "Unsupervised Representation Learning with DCGANs" - DCGAN architecture
- Arjovsky et al. (2017): "Wasserstein GAN" - WGAN and Earth Mover distance
- Gulrajani et al. (2017): "Improved Training of WGANs" - Gradient penalty

**Advanced Architectures:**
- Progressive GAN (2018): Growing resolution during training
- StyleGAN (2019): Style-based generator for high-quality faces
- BigGAN (2019): Large-scale class-conditional generation

**Related Notebooks in This Series:**
- `generation-image.ipynb`: VAE + GAN comparison
- `diffusion-models.ipynb`: Modern diffusion-based generation
- `latent-diffusion.ipynb`: Stable Diffusion architecture